## Appendix: Media Condition Summary
This notebook reads:
- `condition_defs.tsv`
- `media_recipes.tsv`
- linked base media files in `flat/condition/media/`

It shows:
1. Table 1: a summary of 5 media conditions with report-ready condition labels.
2. Appendix note: explanation of each Table 1 column.
3. Tables for each linked base media file.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
from docx import Document


def dataframe_to_docx_table(doc, df, title):
    """Append a dataframe as a formatted table to a Word document."""
    doc.add_heading(title, level=2)
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = 'Table Grid'

    header_cells = table.rows[0].cells
    for i, col in enumerate(df.columns):
        header_cells[i].text = str(col)

    for row in df.fillna('').astype(str).itertuples(index=False):
        row_cells = table.add_row().cells
        for i, value in enumerate(row):
            row_cells[i].text = value


# Paths
cond_dir = Path('/user/home/il22158/work/vEcoli/reconstruction/ecoli/flat/condition')
media_dir = cond_dir / 'media'
condition_defs_path = cond_dir / 'condition_defs.tsv'
media_recipes_path = cond_dir / 'media_recipes.tsv'
output_docx_path = Path('/user/home/il22158/work/vEcoli/ecoli/writing/media_condition_tables.docx')

# Load source tables
condition_defs = pd.read_csv(condition_defs_path, sep='\t', comment='#')
media_recipes = pd.read_csv(media_recipes_path, sep='\t', comment='#')

# Pick 5 conditions shown in condition_defs.tsv
n_conditions = 5
conditions_5 = condition_defs.head(n_conditions).copy()

# Rename condition labels for report-ready naming
condition_label_map = {
    'basal': 'baseline',
    'with_aa': 'with amino-acid',
    'acetate': 'acetate',
    'succinate': 'succinate',
    'no_oxygen': 'no oxygen',
}
conditions_5['condition'] = conditions_5['condition'].map(condition_label_map).fillna(conditions_5['condition'])

# Join condition -> nutrient/media recipe details
summary = conditions_5.merge(
    media_recipes,
    left_on='nutrients',
    right_on='media id',
    how='left'
)

summary_cols = [
    'condition',
    'nutrients',
    'doubling time (units.min)',
    'base media',
    'base media volume (units.L)',
    'added media',
    'added media volume (units.L)',
    'ingredients'
]

display(Markdown('### Table 1. Five Media Conditions'))
display(summary[summary_cols])

# Appendix-ready column explanations for Table 1
column_explanations = pd.DataFrame(
    {
        'Column': summary_cols,
        'Explanation': [
            'Report label of the experimental condition.',
            'Media recipe identifier linked to each condition in condition_defs.tsv.',
            'Expected doubling time under the condition, in minutes.',
            'ID of the linked base medium file in flat/condition/media (without .tsv in this column).',
            'Volume (L) of the base medium used in the recipe.',
            'ID of any additional medium mixed with the base medium.',
            'Volume (L) of the added medium used in the recipe.',
            'Explicit ingredient modifications listed in media_recipes.tsv for the condition recipe.',
        ],
    }
)

display(Markdown('### Appendix Note: Column Definitions for Table 1'))
display(column_explanations)

# Collect linked base media files for these 5 conditions
base_media_ids = (
    summary['base media']
    .dropna()
    .astype(str)
    .str.strip()
)
base_media_ids = sorted([m for m in base_media_ids.unique() if m])

base_media_tables = {}
display(Markdown('### Linked Base Media Tables'))

for base_media_id in base_media_ids:
    base_media_path = media_dir / f'{base_media_id}.tsv'

    if not base_media_path.exists():
        display(Markdown(f'#### {base_media_id} (file missing)'))
        continue

    base_df = pd.read_csv(base_media_path, sep='\t', comment='#')
    base_media_tables[base_media_id] = base_df

    display(Markdown(f'#### {base_media_id}'))
    display(base_df)

# Export all tables to a single Word document
doc = Document()
doc.add_heading('Appendix Tables: Media Conditions', level=1)

dataframe_to_docx_table(doc, summary[summary_cols], 'Table 1. Five Media Conditions')
dataframe_to_docx_table(doc, column_explanations, 'Table 2. Column Definitions for Table 1')

for idx, base_media_id in enumerate(base_media_ids, start=3):
    if base_media_id in base_media_tables:
        dataframe_to_docx_table(doc, base_media_tables[base_media_id], f'Table {idx}. Base Media {base_media_id}')

doc.save(output_docx_path)
display(Markdown(f'### Word file saved: {output_docx_path}'))

### Table 1. Five Media Conditions

,condition,nutrients,doubling time (units.min),base media,base media volume (units.L),added media,added media volume (units.L),ingredients
0,baseline,minimal,44.0,MIX0-57,1.0,NaN,0.0,[]
1,with amino-acid,minimal_plus_amino_acids,25.0,MIX0-57,0.8,5X_supplement_EZ,0.2,[]
2,acetate,minimal_acetate,136.0,MIX0-58,1.0,NaN,0.0,[]
3,succinate,minimal_succinate,82.0,MIX0-844,1.0,NaN,0.0,[]
4,no oxygen,minimal_minus_oxygen,100.0,MIX0-57,1.0,NaN,0.0,"[""OXYGEN-MOLECULE"", ""4FE-4S""]"


### Appendix Note: Column Definitions for Table 1

,Column,Explanation
0,condition,Report label of the experimental condition.
1,nutrients,Media recipe identifier linked to each conditi...
2,doubling time (units.min),"Expected doubling time under the condition, in..."
3,base media,ID of the linked base medium file in flat/cond...
4,base media volume (units.L),Volume (L) of the base medium used in the recipe.
5,added media,ID of any additional medium mixed with the bas...
6,added media volume (units.L),Volume (L) of the added medium used in the rec...
7,ingredients,Explicit ingredient modifications listed in me...


### Linked Base Media Tables

...

#### MIX0-58

,molecule id,concentration (units.mmol / units.L)
0,NA+,135.1500
1,Pi,63.8510
2,CL-,55.3890
3,AMMONIUM,30.2720
4,K+,21.8830
5,SULFATE,15.0770
6,MG+2,2.1000
7,CA+2,0.0900
8,FE+3,0.0031
9,WATER,inf


#### MIX0-844

,molecule id,concentration (units.mmol / units.L)
0,NA+,135.1500
1,Pi,63.8510
2,CL-,55.3890
3,AMMONIUM,30.2720
4,K+,21.8830
5,SULFATE,15.0770
6,MG+2,2.1000
7,CA+2,0.0900
8,FE+3,0.0031
9,WATER,inf


### Word file saved: /user/home/il22158/work/vEcoli/ecoli/writing/media_condition_tables.docx